# 04 · Desafio de Analytics (CineData Analytics)

Consultas sobre o Star Schema da camada Gold.

**Recorte temporal (perguntas 5 e 6):** a data limite superior é a data de lançamento **mais recente** entre os filmes com status
"Lançado" e `data_lancamento <= current_date()` (ignora datas futuras e filmes não lançados). Os "últimos N anos" são contados para trás a
partir dessa data (`add_months(data_limite, -12*N)`).

In [0]:
spark.sql("USE cinedata_analytics.gold")

DataFrame[]

### 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(spark.sql("""
    SELECT CAST(SUM(receita_brl) AS DECIMAL(18,2)) AS receita_total_brl
    FROM fact_movies_performance
"""))

receita_total_brl
874082175327.00


### 2. Quais são os 5 filmes com maior popularidade? Mostre titulo e valor de popularidade.

A popularidade já chega saneada da Silver (valores que eram anos deslocados foram invalidados).

In [0]:
display(spark.sql("""
    SELECT d.titulo, f.popularidade
    FROM fact_movies_performance f
    JOIN dim_movies d ON f.sk_movie_id = d.sk_movie_id
    WHERE f.popularidade IS NOT NULL
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


### 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.

In [0]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
    FROM bridge_movie_genre b
    JOIN dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC, g.nome_genero
"""))

nome_genero,qtd_filmes
Drama,32322
Documentary,19088
Comedy,18644
Thriller,10281
Horror,9740
Romance,7646
Action,6058
Crime,4743
Animation,4478
Tv Movie,4079


### 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).

In [0]:
display(spark.sql("""
    WITH ranking AS (
        SELECT d.titulo,
               f.receita_usd,
               f.receita_brl,
               RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
        FROM fact_movies_performance f
        JOIN dim_movies d ON f.sk_movie_id = d.sk_movie_id
        WHERE f.receita_usd IS NOT NULL
    )
    SELECT posicao_ranking, titulo, receita_usd, receita_brl
    FROM ranking
    WHERE posicao_ranking <= 10
    ORDER BY posicao_ranking, titulo
"""))

posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,15120000000.00
2,Avatar: The Way of Water,2320250281.00,12529351517.40
3,AVENGERS: INFINITY WAR,2052415039.00,11083041210.60
4,spider-man: no way home,1921847111.00,10377974399.40
5,The Lion King,1663075401.00,8980607165.40
6,Top Gun: Maverick,1488732821.00,8039157233.40
7,Barbie,1428545028.00,7714143151.20
8,The Super Mario Bros. Movie,1355725263.00,7320916420.20
9,Black Panther,1349926083.00,7289600848.20
10,Star Wars: The Last Jedi,1332698830.00,7196573682.00


### 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

Em caso de empate no topo, todos os atores empatados são listados.

In [0]:
display(spark.sql("""
    WITH data_limite AS (
        SELECT MAX(data_lancamento) AS dt_max
        FROM dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento <= current_date()
    ),
    participacoes AS (
        SELECT p.nome_pessoa AS ator,
               COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes
        FROM dim_movies m
        CROSS JOIN data_limite l
        JOIN bridge_movie_person b ON b.sk_movie_id = m.sk_movie_id
        JOIN dim_people p          ON p.sk_person_id = b.sk_person_id
        WHERE p.tipo_pessoa = 'Ator'
          AND m.status_filme = 'Lançado'
          AND m.data_lancamento >= add_months(l.dt_max, -24)
          AND m.data_lancamento <= l.dt_max
        GROUP BY p.nome_pessoa
    )
    SELECT ator, qtd_filmes
    FROM participacoes
    WHERE qtd_filmes = (SELECT MAX(qtd_filmes) FROM participacoes)
    ORDER BY ator
"""))

ator,qtd_filmes
Kevin Hart,64


### 6. Qual a produtora de filmes teve o maior lucro nos últimos 5 anos?

O lucro de cada filme é atribuído integralmente a cada uma das produtoras do filme (a base não informa a divisão de participação).

In [0]:
display(spark.sql("""
    WITH data_limite AS (
        SELECT MAX(data_lancamento) AS dt_max
        FROM dim_movies
        WHERE status_filme = 'Lançado'
          AND data_lancamento <= current_date()
    )
    SELECT c.nome_produtora,
           CAST(SUM(f.lucro_usd) AS DECIMAL(18,2)) AS lucro_total_usd,
           CAST(SUM(f.lucro_brl) AS DECIMAL(18,2)) AS lucro_total_brl,
           COUNT(DISTINCT m.sk_movie_id)           AS qtd_filmes
    FROM fact_movies_performance f
    JOIN dim_movies m             ON m.sk_movie_id = f.sk_movie_id
    CROSS JOIN data_limite l
    JOIN bridge_movie_company b   ON b.sk_movie_id = m.sk_movie_id
    JOIN dim_companies c          ON c.sk_company_id = b.sk_company_id
    WHERE f.lucro_usd IS NOT NULL
      AND m.data_lancamento >= add_months(l.dt_max, -60)
      AND m.data_lancamento <= l.dt_max
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 1
"""))

nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes
Universal Pictures,5430017131.00,29322092507.40,38
